# Análisis de Mañaneras — Embeddings semánticos
### Menciones indirectas por similitud de significado

> **Solo se necesita editar la sección de CONFIGURACIÓN.**
> Este notebook está pensado para correr en un entorno virtual local. Requiere las dependencias extra de `requirements-embeddings.txt` (ver `README.md`).
>
> A diferencia de `Analisis_Conteo.ipynb` (coincidencia literal de texto), aquí se detectan fragmentos que **hablan de un grupo aunque no usen las palabras exactas**, comparando el significado de fragmentos del texto contra frases descriptivas de cada grupo.
>
> ⚠️ La primera ejecución descarga el modelo de embeddings (~470 MB) desde Hugging Face; necesitas conexión a internet la primera vez.

---
## CONFIGURACIÓN

In [ ]:
# =============================================================
# CONFIGURACIÓN
# =============================================================

# Carpeta local donde están guardados tus archivos .docx
# Los archivos deben incluir en su nombre el mes (3 letras) y el año, ej: "ene-2019_mananera.docx"
RUTA_DATOS = "./data/mananeras"

# Rango de fechas del análisis
FECHA_INICIO = "2018-12-01"   # formato: AAAA-MM-DD
FECHA_FIN    = "2026-01-31"   # formato: AAAA-MM-DD

# Corte entre presidencias
FECHA_CAMBIO_PRESIDENTE = "2024-10-01"

# =============================================================
# GRUPOS Y FRASES DESCRIPTIVAS PARA EMBEDDINGS SEMÁNTICOS
# Cada grupo se representa por el promedio de los vectores de sus frases.
# =============================================================
GRUPOS_EMBEDDINGS = {
    "Empresarios": [
        "empresarios y sector privado mexicano",
        "iniciativa privada y COPARMEX",
        "inversión privada y cámaras empresariales",
        "grandes corporativos y negocios en México"
    ],
    "Periodistas": [
        "periodistas y prensa libre en México",
        "medios de comunicación y reporteros",
        "libertad de prensa y censura mediática",
        "comunicadores y corresponsales de medios"
    ],
    "Ambientalistas": [
        "ambientalistas y ecologistas mexicanos",
        "protección del medio ambiente y sustentabilidad",
        "cambio climático y activismo ambiental",
        "derechos ambientales y conservación de ecosistemas"
    ],

    # "Nombre del grupo": [
    #     "frase descriptiva 1",
    #     "frase descriptiva 2",
    # ],
}

# Umbral de similitud coseno para considerar un fragmento "relevante"
UMBRAL_SIMILITUD = 0.6  # Por determinar

print("✅ Config guardada.")
print(f"   Período: {FECHA_INICIO} → {FECHA_FIN}")
print(f"   Grupos: {list(GRUPOS_EMBEDDINGS.keys())}")
print(f"   Umbral semántico: {UMBRAL_SIMILITUD}")


---
## CÓDIGO BASE — No modificar

### Carga de archivos locales (.docx)

In [ ]:
import os, re
from datetime import datetime
from docx import Document
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Parsear fechas de configuración
_fecha_inicio = datetime.strptime(FECHA_INICIO, "%Y-%m-%d")
_fecha_fin    = datetime.strptime(FECHA_FIN,    "%Y-%m-%d")
_fecha_cambio = datetime.strptime(FECHA_CAMBIO_PRESIDENTE, "%Y-%m-%d")

# diccionario aux meses
_MESES_ES = {
    "ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12
}

def _extraer_fecha(nombre):
    m = re.search(r'([A-Za-z]{3})[-_](\d{4})', nombre)
    if m:
        mes_str = m.group(1).upper()
        año = int(m.group(2))
        mes = _MESES_ES.get(mes_str)
        if mes:
            return datetime(año, mes, 1)
    return None

def _cargar_corpus():
    if not os.path.exists(RUTA_DATOS):
        print(f"⚠️  Carpeta no encontrada: {RUTA_DATOS}")
        print("   Verifica que RUTA_DATOS apunte a la carpeta correcta en tu computadora.")
        return []
    archivos = []
    sin_fecha = []
    for nombre in sorted(os.listdir(RUTA_DATOS)):
        if not nombre.lower().endswith('.docx'):
            continue
        fecha = _extraer_fecha(nombre)
        if not fecha:
            sin_fecha.append(nombre)
            continue
        if not (_fecha_inicio <= fecha <= _fecha_fin):
            continue
        ruta = os.path.join(RUTA_DATOS, nombre)
        try:
            doc = Document(ruta)
            texto = ' '.join(p.text.strip() for p in doc.paragraphs if p.text.strip())
            presidente = "AMLO" if (fecha.year, fecha.month) < (_fecha_cambio.year, _fecha_cambio.month) else "Claudia Sheinbaum"
            archivos.append({
                'fecha':      fecha,
                'mes':        fecha.strftime('%Y-%m'),
                'año':        fecha.year,
                'presidente': presidente,
                'texto':      texto,
                'n_palabras': len(texto.split())
            })
        except Exception as e:
            print(f"   ⚠️  Error leyendo {nombre}: {e}")

    print(f"✅ {len(archivos)} archivos cargados")
    print(f"   Período: {_fecha_inicio.strftime('%b %Y')} → {_fecha_fin.strftime('%b %Y')}")
    dist = pd.Series([d['presidente'] for d in archivos]).value_counts()
    for p, n in dist.items():
        print(f"   {p}: {n} archivos")
    if sin_fecha:
        print(f"\n   ⚠️  {len(sin_fecha)} archivos ignorados (no se pudo leer fecha del nombre):")
        for f in sin_fecha[:5]:
            print(f"      {f}")
    return sorted(archivos, key=lambda x: x['fecha'])

_corpus = _cargar_corpus()


---
## Análisis semántico con embeddings

Captura menciones indirectas: fragmentos que hablan de un grupo aunque no usen las palabras exactas.
Genera tres visualizaciones: comparación entre presidentes, línea de tiempo mensual y heatmap trimestral.

In [ ]:
# ============================================================
#  ANÁLISIS SEMÁNTICO CON EMBEDDINGS
# ============================================================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

if not _corpus:
    print("No hay archivos cargados. Revisa RUTA_DATOS y vuelve a ejecutar la celda de carga.")
else:
    _COLORES_P = {"AMLO": "#1A3A5C", "Claudia Sheinbaum": "#C0392B"}
    _PALETA_E  = sns.color_palette("muted", len(GRUPOS_EMBEDDINGS))
    _grupos_e  = list(GRUPOS_EMBEDDINGS.keys())

    # ── Cargar modelo multilingüe ─────────────────────────────
    print("Cargando modelo de embeddings (paraphrase-multilingual-MiniLM-L12-v2)...")
    _modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("✅ Modelo listo.\n")

    # ── Embedding de cada grupo (promedio de sus frases) ──────
    print("Generando vectores de grupos de interés...")
    _emb_grupos = {}
    for grupo, frases in GRUPOS_EMBEDDINGS.items():
        vecs = _modelo.encode(frases, show_progress_bar=False)
        _emb_grupos[grupo] = np.mean(vecs, axis=0)
        print(f"   ✓ {grupo}")

    # ── Dividir texto en fragmentos de ~3 oraciones ───────────
    def _chunking(texto, n=3):
        oraciones = [s.strip() for s in re.split(r'(?<=[.!?])\s+', texto) if len(s.strip()) > 40]
        return [' '.join(oraciones[i:i+n]) for i in range(0, len(oraciones), n)
                if len(' '.join(oraciones[i:i+n])) > 60]

    # ── Procesar corpus ───────────────────────────────────────
    print(f"\nProcesando {len(_corpus)} archivos...")
    print(f"Umbral de similitud: {UMBRAL_SIMILITUD}\n")
    filas_e = []

    for i, doc in enumerate(_corpus):
        if (i+1) % 30 == 0 or i == 0:
            print(f"   Archivo {i+1}/{len(_corpus)} — {doc['fecha'].strftime('%b %Y')} ({doc['presidente']})")
        chunks = _chunking(doc['texto'])
        if not chunks:
            continue
        emb_chunks = _modelo.encode(chunks, batch_size=64, show_progress_bar=False)

        for grupo, emb_g in _emb_grupos.items():
            sims = cosine_similarity([emb_g], emb_chunks)[0]
            relevantes = int((sims >= UMBRAL_SIMILITUD).sum())
            filas_e.append({
                'fecha':        doc['fecha'],
                'mes':          doc['mes'],
                'presidente':   doc['presidente'],
                'grupo':        grupo,
                'relevantes':   relevantes,
                'sim_max':      round(float(sims.max()), 3) if len(sims) > 0 else 0.0,
                'n_chunks':     len(chunks),
                'cobertura_pct': round(relevantes / len(chunks) * 100, 2) if chunks else 0
            })

    df_e = pd.DataFrame(filas_e)
    print("\n✅ Embeddings calculados.\n")

    # ── Tabla resumen ──────────────────────────────────────────
    print("═"*62)
    print("  FRAGMENTOS RELEVANTES TOTALES POR PRESIDENTE Y GRUPO")
    print("═"*62)
    res_e = df_e.groupby(['presidente','grupo'])['relevantes'].sum().unstack(fill_value=0)
    print(res_e.to_string())
    print("═"*62)
    print("\n  (Cobertura media: % de fragmentos por encima del umbral)")
    print("═"*62)
    res_pct = df_e.groupby(['presidente','grupo'])['cobertura_pct'].mean().round(2).unstack(fill_value=0)
    print(res_pct.to_string())
    print("═"*62)

    # ── VIZ 1: Barras por presidente ──────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot_e = df_e.groupby(['presidente','grupo'])['cobertura_pct'].mean().unstack()
    pivot_e.plot(kind='bar', ax=ax, color=_PALETA_E, width=0.55, edgecolor='white')
    ax.set_title(
        f"Cobertura semántica por presidente y grupo\n(umbral de similitud: {UMBRAL_SIMILITUD})",
        fontsize=13, pad=14, fontweight='bold')
    ax.set_xlabel("", fontsize=1)
    ax.set_ylabel("% de fragmentos relevantes", fontsize=10)
    ax.set_xticklabels(pivot_e.index, rotation=0, fontsize=11)
    ax.legend(title="Grupo de interés", bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig("embeddings_presidentes.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("\n Gráfica 1 guardada: embeddings_presidentes.png")

    # ── VIZ 2: Línea de tiempo mensual ────────────────────────
    df_et = df_e.groupby(['mes','grupo','presidente'])['cobertura_pct'].mean().reset_index()
    df_et['mes_dt'] = pd.to_datetime(df_et['mes'])

    fig, axes = plt.subplots(len(_grupos_e), 1, figsize=(14, 4*len(_grupos_e)), sharex=True)
    axes = [axes] if len(_grupos_e) == 1 else axes

    for ax, (grupo, color) in zip(axes, zip(_grupos_e, _PALETA_E)):
        dg = df_et[df_et['grupo'] == grupo]
        for pres, pc in _COLORES_P.items():
            dp = dg[dg['presidente'] == pres].sort_values('mes_dt')
            if not dp.empty:
                ax.plot(dp['mes_dt'], dp['cobertura_pct'], color=pc, label=pres, linewidth=2, alpha=0.9)
                ax.fill_between(dp['mes_dt'], dp['cobertura_pct'], alpha=0.08, color=pc)
        ax.set_title(f"{grupo} — semántico", fontsize=11, fontweight='bold', loc='left', pad=6)
        ax.set_ylabel("% cobertura\nsemántica", fontsize=8)
        ax.legend(fontsize=8, framealpha=0.6)
        ax.grid(axis='y', alpha=0.2, linestyle='--')
        ax.spines[['top','right']].set_visible(False)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
    fig.suptitle("Evolución semántica mensual por grupo", fontsize=14, y=1.01, fontweight='bold')
    plt.tight_layout()
    plt.savefig("embeddings_timeline.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(" Gráfica 2 guardada: embeddings_timeline.png")

    # ── VIZ 3: Heatmap trimestral semántico ───────────────────
    df_eq = df_e.groupby(['mes','grupo'])['cobertura_pct'].mean().unstack(fill_value=0)
    df_eq.index = pd.to_datetime(df_eq.index)
    df_eq = df_eq.resample('QE').mean()  # 'QE' = fin de trimestre (pandas >= 2.2)
    df_eq.index = df_eq.index.to_period('Q').astype(str)

    fig, ax = plt.subplots(figsize=(max(10, len(df_eq)*0.5), max(4, len(_grupos_e)*1.2)))
    sns.heatmap(df_eq.T, cmap="PuBu", linewidths=0.4, linecolor='white',
                annot=True, fmt='.1f', annot_kws={'size': 7},
                ax=ax, cbar_kws={'label': '% cobertura semántica', 'shrink': 0.6})
    ax.set_title(
        f"Heatmap semántico por trimestre y grupo  (umbral: {UMBRAL_SIMILITUD})",
        fontsize=12, pad=12, fontweight='bold')
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=50, ha='right', fontsize=7)
    plt.tight_layout()
    plt.savefig("embeddings_heatmap.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(" Gráfica 3 guardada: embeddings_heatmap.png")

    print(f"\n✅ Análisis semántico completado — umbral usado: {UMBRAL_SIMILITUD}")
    print("   Archivos PNG disponibles en la carpeta del proyecto")
